# Test Stage 14a — Does the board's own clock hold still?

Every frequency this system reports is scaled by its 125 MHz clock. If that clock is wrong,
every frequency is wrong by the same fraction, and nothing measured so far would notice.

**You cannot measure a clock's accuracy against itself.** But accuracy is not the only question,
and it is not even the important one here.

### Accuracy and stability are different, and only one needs a standard

| | What it is | Can it be measured here? |
|---|---|---|
| **Accuracy** | A fixed offset. Both crystals 20 ppm low and the ratio is perfect while everything is 20 ppm wrong | **No.** Needs an outside source — Stage 14b |
| **Stability** | Whether it wanders, and how much over a given time | **Yes**, and this notebook does it |

### Why stability is measurable without a standard

This board carries **two independent crystals**: the 125 MHz oscillator driving the fabric and
the sample counter, and the processor's own crystal, which is what Linux's clock runs on.
Neither is a standard, but they are independent — and independent crystals of different cuts do
not drift together. So a steady ratio between them means both are steady, and a drifting ratio
means at least one is drifting by that much.

Stage 2 already makes this comparison, for 0.3 s, as a sanity check. Run it long enough and it
becomes a measurement:

| Run length | Samples counted | Resolution |
|---|---|---|
| 0.3 s (Stage 2) | 37,000 | 27 ppm |
| 100 s | 1.2 × 10⁷ | 0.08 ppm |
| 1000 s | 1.2 × 10⁸ | 0.008 ppm |

### Which of your measurements actually care about accuracy

| Quantity | Sensitive to clock accuracy? |
|---|---|
| Q = centre ÷ linewidth | **No** — a ratio of two frequencies from the same clock; it cancels exactly |
| Linewidth in Hz | Scales with it: 10 ppm of 222 kHz is 2 Hz. Irrelevant |
| Decoupling shift | A difference: 10 ppm of 170 kHz is 1.7 Hz. Irrelevant |
| Oscillation frequency scatter | **No** — a scatter is a stability measurement |
| Absolute centre frequency | **Yes** — 10 ppm of 40 MHz is 400 Hz |

Only the last one cares. The paper's 0.75 Hz figure is a scatter, not an absolute — so for
nearly everything measured here, stability is the question, and stability is what this
notebook settles.

---

### Simulation or the real trap?

**Unchanged, and it does not involve the coil at all.** Run it whenever convenient; the answer
is a property of the board.

## What you need

| | |
|---|---|
| Stages 1–7 | Passing |
| Time | About 25 minutes, nearly all of it waiting |
| Cables | **None** |

## Setup — drivers and capture

In [ ]:
import os, mmap, struct, time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

FS_CLK = 125e6
F0     = 40.0e6          # the coil being emulated: ~40 MHz
Q_SET  = 180.0           # ~180 at room temperature
SETTLE = 0.02
N_AVG  = 32

# =============================================================================
#  Environment shims. This board carries an older numpy and matplotlib than a
#  current desktop, and two conveniences used below arrived later than what is
#  installed here: nan_to_num's `nan=` keyword needs numpy >= 1.17, and
#  Axes.set_facecolor needs matplotlib >= 2.0. Provided once, rather than as
#  version tests scattered through the analysis cells.
# =============================================================================
def nan_fill(a, value):
    """np.nan_to_num(a, nan=value), for numpy versions without the keyword."""
    out = np.array(a, dtype=float, copy=True)
    out[~np.isfinite(out)] = value
    return out


def set_bg(ax, colour):
    """Axes background colour, across the matplotlib 1.x -> 2.x rename."""
    if hasattr(ax, 'set_facecolor'):
        ax.set_facecolor(colour)
    else:
        ax.set_axis_bgcolor(colour)


# =============================================================================
#  Drivers - the Stage 8 set, with the resonator gaining its two noise knobs.
#  CRITICAL RULE throughout: one 32-bit register access at a time.
# =============================================================================
class Bus(object):
    def __init__(self, base, span=0x1000):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, span, offset=base)
    def w(self, off, val):
        self.m[off:off + 4] = struct.pack('<I', val & 0xFFFFFFFF)
    def r(self, off):
        return struct.unpack('<I', self.m[off:off + 4])[0]

def s32(v):
    return v - (1 << 32) if v & 0x80000000 else v


class Acq(Bus):
    """Acquisition system as narrowband voltmeter (PEEK at DC) and as
    recorder (capture to memory - defined in the next cell)."""
    def __init__(self):
        Bus.__init__(self, 0x40600000)
        self.dec = 10
    def start(self, dec_log2=10):
        self.dec = dec_log2
        self.w(0x00, 0)
        self.w(0x08, dec_log2)
        self.w(0x0C, 0)
        self.w(0x00, 1 | 2)
        self.w(0x00, 1)
    def tune(self, f):
        self.w(0x04, int(round(f / FS_CLK * (1 << 32))) & 0xFFFFFFFF)
    def measure(self, navg=N_AVG):
        zi = 0.0; zq = 0.0
        for _ in range(navg):
            zi += s32(self.r(0x18))
            zq += s32(self.r(0x1C))
        return complex(zi / navg, zq / navg)
    def f_out(self):
        return FS_CLK / (1 << self.dec)


class Resonator(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40400000)
        self.f0_q = None; self.q_q = None
    def config(self, f0, Q, delay=15, oshift=7):
        w0 = 2.0 * np.pi * f0 / FS_CLK
        r  = 1.0 - np.pi * (f0 / FS_CLK) / Q
        ci = int(round(r * np.cos(w0) * 131072))
        cs = int(round(r * np.sin(w0) * 131072))
        self.w(0x04, ci & 0x3FFFF)
        self.w(0x08, cs & 0x3FFFF)
        self.w(0x0C, delay)
        self.w(0x10, oshift)
        rq        = np.hypot(ci, cs) / 131072.0
        self.f0_q = np.arctan2(cs, ci) / (2.0 * np.pi) * FS_CLK
        self.q_q  = np.pi * (self.f0_q / FS_CLK) / (1.0 - rq)
    def shift_f0(self, df, Q=None):
        """Retune the emulated coil by df hertz - the drift the servo fights.
        One coefficient rewrite; Q is preserved."""
        self.config(self.f0_q + df, Q if Q is not None else self.q_q,
                    delay=15, oshift=7)
    # --- the two noise knobs (registers added for this stage) -------------
    def noise_tank(self, shift=None):
        """Johnson noise INTO the loop -> Lorentzian bump. None = off.
        Each shift step is 6 dB down from raw (~9460 counts rms)."""
        self.w(0x20, 0 if shift is None else (0x100 | (shift & 0x1F)))
    def noise_out(self, shift=None):
        """Amplifier noise AFTER the loop -> flat floor. None = off."""
        self.w(0x24, 0 if shift is None else (0x100 | (shift & 0x1F)))
    def enable(self):  self.w(0x00, 1)
    def disable(self): self.w(0x00, 0)
    def clear(self):   self.w(0x00, 3)
    def clipped(self): return bool(self.r(0x14) & 1)
    def peek(self):    return s32(self.r(0x18)), s32(self.r(0x1C))


class PhaseShift(Bus):
    LATENCY = 2
    COEF_FS = 16384.0                 # coefficient full scale -> gain LSB
    def __init__(self):
        Bus.__init__(self, 0x40300000)
    def config_delay(self, f):
        best = None
        for k in range(0, 9):
            D = FS_CLK / (4.0 * f) + k * FS_CLK / f
            if D < 1.0 or D > 1000:
                continue
            frac  = D - np.floor(D)
            score = min(frac, 1.0 - frac)
            if best is None or score < best[0]:
                best = (score, int(np.floor(D)), frac)
        self.w(0x08, best[1])
        self.w(0x0C, int(round(best[2] * 16384)))
        self.w(0x10, 0)
        return best[1], best[2]
    def set(self, A, phi_deg, f=F0):
        phi = phi_deg - 360.0 * self.LATENCY * f / FS_CLK
        self.w(0x00, int(round(A * np.cos(np.radians(phi)) * self.COEF_FS)) & 0xFFFF)
        self.w(0x04, int(round(A * np.sin(np.radians(phi)) * self.COEF_FS)) & 0xFFFF)
    def gain_lsb(self, phi_deg, f=F0):
        """Smallest change in A this actuator can express at this phase. The
        two coefficients quantise independently, so the step in |A| is set by
        the LARGER of the two projections. Step 5c compares this against the
        width of the band a software loop would need to steer within."""
        phi = np.radians(phi_deg - 360.0 * self.LATENCY * f / FS_CLK)
        return 1.0 / (self.COEF_FS * max(abs(np.cos(phi)), abs(np.sin(phi))))
    def off(self):
        self.set(0.0, 0.0)


class Generator(Bus):
    def __init__(self):
        Bus.__init__(self, 0x40200000, span=0x20000)
    def load_sine(self):
        for i in range(16384):
            v = int(round(8191.0 * np.sin(2.0 * np.pi * i / 16384.0)))
            self.w(0x10000 + 4 * i, v & 0x3FFF)
        self.w(0x08, (16384 << 16) - 1)
        self.w(0x0C, 0)
    def amp(self, frac):
        self.w(0x04, int(round(frac * 8192)) & 0x3FFF)
    def freq(self, f):
        self.w(0x10, int(round(f / FS_CLK * (1 << 30))))
    def run(self):
        self.w(0x00, 0x40)
        self.w(0x00, 0x11)
    def stop(self):
        self.amp(0.0)
        self.w(0x00, 0xC0)


acq, reso, ps, gen = Acq(), Resonator(), PhaseShift(), Generator()
print('drivers ready  (numpy %s, matplotlib %s)'
      % (np.__version__, matplotlib.__version__))

In [ ]:
# =============================================================================
#  Memory capture - the Stage 6/7 system, condensed. Noise SPECTRA need time
#  records, and time records need memory; the PEEK voltmeter cannot see a
#  spectrum. Same safeguards as Stage 7: region from the device tree, stop
#  address inclusive, poison pre-fill, write-error and poison checks that
#  RAISE rather than return plausible garbage. (Bitstreams from the Stage 8
#  rebuild onward carry the trusted badge, so TrustZone no longer matters -
#  but the checks stay, because they cost nothing and catch everything.)
# =============================================================================
def _be32(b, i=0):
    return struct.unpack('>I', b[i * 4:i * 4 + 4])[0]

def _dt_read(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        try:
            with open(os.path.join(root, path), 'rb') as fh:
                return fh.read()
        except (IOError, OSError):
            pass
    return None

blob = _dt_read('memory/reg') or _dt_read('memory@0/reg')
assert blob and len(blob) >= 8, 'cannot read the device tree memory node'
declared_end = _be32(blob, 0) + _be32(blob, 1)
assert declared_end < 512 * 1024**2, 'no memory above the declared end - see Stage 6 Step 1'
MEM_START = (declared_end + 0xFFF) & ~0xFFF
MEM_STOP  = MEM_START + (0x00800000 & ~7) - 8       # LAST address, inclusive
POISON    = 0xDEADBEEF
print('capture region 0x%08X .. 0x%08X inclusive' % (MEM_START, MEM_STOP + 7))


CAPTURE_TRACE = None      # set to a path (e.g. '/tmp/capture.log') to log every
                          # capture to disk before it runs


def _trace(msg):
    """Append to the trace file and force it to disk.

    A dead kernel takes buffered stdout with it, which is why a crash inside
    capture() leaves the notebook showing nothing at all. This survives,
    so the last line in the file names the call that killed it."""
    if not CAPTURE_TRACE:
        return
    with open(CAPTURE_TRACE, 'a') as fh:
        fh.write('%.3f %s\n' % (time.time(), msg))
        fh.flush()
        os.fsync(fh.fileno())


def _poison(nbytes):
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, nbytes, offset=MEM_START)
    try:
        mm[0:nbytes] = np.full(nbytes // 4, POISON, dtype='<u4').tobytes()
    finally:
        mm.close(); os.close(fd)


def capture(f_lo, dec_log2, n_samples, settle=SETTLE):
    """Record n_samples of the complex stream; raises on any transport fault."""
    _trace('capture f=%.1f dec=%d n=%d' % (f_lo, dec_log2, n_samples))
    _poison(n_samples * 8)
    acq.w(0x00, 0)
    acq.w(0x08, dec_log2); acq.dec = dec_log2
    acq.w(0x0C, 0)
    acq.tune(f_lo)
    acq.w(0x28, MEM_START)
    acq.w(0x2C, MEM_STOP)
    # Read the bounds back BEFORE arming. The writer is a DMA master: if the
    # stop bound did not take - wrong bitstream, unimplemented register, a
    # write that silently went nowhere - it runs from MEM_START upward through
    # whatever follows, and what follows is the memory Linux is using. That is
    # not an exception with a traceback to read; it is a dead kernel. Two
    # register reads cost microseconds and turn the worst failure mode in this
    # notebook into a sentence.
    lo_rb, hi_rb = acq.r(0x28), acq.r(0x2C)
    if lo_rb != MEM_START or hi_rb != MEM_STOP:
        acq.w(0x00, 0)                              # ensure nothing is armed
        raise RuntimeError(
            'capture bounds did not take: wrote 0x%08X..0x%08X, read back '
            '0x%08X..0x%08X. Refusing to arm an unbounded DMA writer.'
            % (MEM_START, MEM_STOP, lo_rb, hi_rb))
    if n_samples * 8 > (MEM_STOP + 8 - MEM_START):
        raise ValueError('record of %d samples (%d bytes) exceeds the %d-byte '
                         'capture region'
                         % (n_samples, n_samples * 8, MEM_STOP + 8 - MEM_START))
    # SETTLE FIRST, RECORD SECOND. Arming immediately and sleeping afterwards
    # would let the record's opening milliseconds hold the filter pipeline still
    # flushing the PREVIOUS tuning - which in a sweep poisons each point with its
    # predecessor's power, most visibly the first point, and can drag a fit wide
    # by tens of percent from the far end of the grid. Instead the chain
    # free-runs at the new tuning while the pipeline flushes, and only then does
    # the writer start: sample 0 of every record is already settled.
    acq.w(0x00, 1 | 2)                              # run at the new tuning...
    acq.w(0x00, 1)
    time.sleep(settle)                              # ...flush the pipeline...
    acq.w(0x00, 1 | 16 | 128)                       # ...then arm, fresh writer
    acq.w(0x00, 1 | 16)
    time.sleep(n_samples * (1 << dec_log2) / FS_CLK + 0.005)
    acq.w(0x00, 1)                                  # disarm

    st  = acq.r(0x38)
    if st & 8:
        raise RuntimeError('memory refused writes - see Stage 6; run check_system.py')
    got = min(int(acq.r(0x34)), n_samples)
    # try/finally, because everything between the open and the close can raise:
    # mmap itself, the slice (SIGBUS on a bad physical address), an allocation
    # failure on a tight board. Without it each failure leaks a file descriptor
    # AND a mapping of /dev/mem - and Stage 10's scan loop swallows per-point
    # exceptions by design, so the leak would repeat silently across hundreds
    # of grid points until the process runs out of descriptors or address
    # space. That surfaces later as an unrelated-looking out-of-memory error.
    fd = mm = None
    try:
        fd  = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        mm  = mmap.mmap(fd, ((got * 8 + 8191) // 4096) * 4096, offset=MEM_START)
        buf = mm[0:got * 8]
    finally:
        if mm is not None:
            mm.close()
        if fd is not None:
            os.close(fd)
    u32 = np.frombuffer(buf, dtype='<u4')
    if got and int(np.count_nonzero(u32 == POISON)) > max(8, 0.001 * u32.size):
        raise RuntimeError('unwritten words in the record - transport fault, see Stage 6')
    raw = np.frombuffer(buf, dtype='<i4').astype(np.float64)
    acq.w(0x00, 1)                                  # back to free-running voltmeter
    _trace('   ok got=%d' % got)
    return raw[0::2] + 1j * raw[1::2]


def welch_psd(z, seg=2048):
    """Averaged power spectrum of a complex record - the standard way to see
    noise: many short transforms averaged beat one long one, because noise in
    a single transform is 100% uncertain no matter how long it is."""
    k = len(z) // seg
    w = np.hanning(seg)
    p = np.zeros(seg)
    for i in range(k):
        p += np.abs(np.fft.fft(z[i * seg:(i + 1) * seg] * w)) ** 2
    f = np.fft.fftfreq(seg, 1.0 / acq.f_out())
    order = np.argsort(f)
    return f[order], p[order] / max(k, 1)


def _lorentz_core(f, p, guard=0.05, minpts=3):
    """Centre, width, peak and BASELINE of a Lorentzian power curve.

    1/p is a parabola in f only for a pure Lorentzian. Any additive floor -
    detector noise, an amplifier pedestal - is absorbed into that parabola and
    inflates the fitted width. That is negligible where the peak towers over
    the floor and severe where it does not, because peak power falls as the
    square of the Q ratio: a floor worth 2% of the open-loop peak is worth 32%
    of a 4x-broadened one. Uncorrected bias, measured: +18% in linewidth at
    Q_eff/Q0 = 1, +26% at 0.5, +153% at 0.25.

    The floor is found by search, not from the wings. For each trial floor:
    subtract it, fit, rebuild the full Lorentzian-plus-floor, and keep the
    trial that reproduces the measured points best. A wing estimate needs
    points beyond ~1.5 linewidths, which a broad line does not have on this
    grid - exactly the case that needs correcting most. The search uses every
    point, so it works at any width. Two passes: coarse over [0, min(p)], then
    fine around the winner.

    Returns (f0, linewidth, peak_above_baseline, baseline).
    """
    f = np.asarray(f, dtype=float)
    p = np.asarray(p, dtype=float)
    pmax = float(np.max(p))
    u = f - f.mean()

    def _try(b):
        q = p - b
        keep = q > guard * q.max()
        if int(np.count_nonzero(keep)) < minpts:
            return None
        A, B, C = np.polyfit(u[keep], 1.0 / q[keep], 2, w=q[keep])
        if A <= 0.0:
            return None                    # parabola opens the wrong way
        u0 = -B / (2.0 * A)
        lw = 2.0 * np.sqrt(max(C / A - u0 ** 2, 1e-30))
        p0 = 1.0 / max(C - A * u0 ** 2, 1e-300)
        f0 = f.mean() + u0
        mod = p0 / (1.0 + (2.0 * (f - f0) / lw) ** 2) + b
        return float(np.sqrt(np.mean(((p - mod) / pmax) ** 2))), (f0, lw, p0, b)

    hi = 0.999 * float(np.min(p))
    best, best_b = _try(0.0), 0.0           # b = 0 is the uncorrected fit
    if hi > 0.0:
        for b in np.linspace(0.0, hi, 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
        w = hi / 16.0
        for b in np.linspace(max(best_b - w, 0.0), min(best_b + w, hi), 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
    if best is None:
        raise ValueError('fit window too narrow: %d points'
                         % int(np.count_nonzero(p > guard * p.max())))
    f0, lw, p0, base = best[1]
    return f0, lw, p0, base


def fit_lorentzian_psd(f, p, guard=0.05):
    """Centre, width from a power spectrum: 1/p is a parabola (Stage 8), after
    the noise floor has been taken out of p."""
    f0, lw, _, _ = _lorentz_core(f, p, guard=guard, minpts=3)
    return f0, lw

In [ ]:
# ============================ CONFIGURATION ==================================
EMULATION  = True    # True: digital resonator, known answers, PASS gates.
                     # False: the real coil - same scan, gates become measurements

TANK_SH    = 3       # tank (Johnson) noise level, 6 dB per step
OUT_SH     = 4       # amplifier-floor level (off during sweeps automatically)

# The gain axis must BRACKET the self-oscillation boundary, or there is no
# boundary in the atlas and every stability margin the designer prints is
# meaningless. K rises ~0.73 per unit A, so onset is near A = 1.4 (+3 dB).
GAINS_DB   = np.arange(-26.0, 6.01, 2.0)     # feedback gain axis (dB of A)
PHIS       = np.arange(0.0, 360.0, 15.0)     # feedback phase axis: 24 points
SWEEP_PTS  = 15      # frequencies per mini-sweep at each grid point
SWEEP_SPAN = 2.5     # mini-sweep half-width in open-loop linewidths
SWEEP_N    = 2048    # samples per frequency point (halved at deep decimations)
SETTLE     = 0.02

# A point is OSCILLATING if, with the noise drive removed, it still holds this
# fraction of its driven power. A ratio, so it needs no absolute floor - on a
# digital plant the undriven floor is exactly zero.
SUS_OSC    = 0.15
DRIVE_STEP = 2       # noise-shift steps (12 dB) used to test loop linearity
# Grid cost: 17 x 24 points x ~1 s. About 7 minutes.
# =============================================================================

def coherence(z):
    """1 for pure noise, 0 for a pure tone; equals 1 - f^2 where f is the
    power fraction carried by a coherent tone. Reported as a diagnostic.
    Not used to classify points: for narrowband noise the effective sample
    count is small and this scatters ~30%, far too coarse to detect anything
    subtle."""
    p = np.abs(z) ** 2
    m = float(np.mean(p))
    return float(np.var(p)) / max(m * m, 1e-300)


def tone_fraction(R):
    return float(np.sqrt(max(0.0, 1.0 - R)))

---\n## Step 1 — The two crystals, compared

In [ ]:
# The two clocks:
#   - the acquisition sample counter, driven by the 125 MHz fabric oscillator
#   - time.time(), driven by the processor's own crystal
# Counting one against the other measures their RATIO. Neither is a standard,
# so this cannot say which is right - only whether they agree, and how steadily.
R_SAMPLE_CNT = 0x34

def sample_count():
    return acq.r(R_SAMPLE_CNT) & 0xFFFFFFFF


def ratio_over(seconds, dec=10):
    """Fractional difference between the two clocks over one interval.

    Returns (measured rate, expected rate, fractional error). The counter is
    32 bits; at the rate used here it takes hours to wrap, and the mask makes
    that explicit rather than assumed."""
    acq.start(dec_log2=dec)
    acq.w(0x00, 1 | 16 | 128)          # reset the writer and arm, so it counts
    acq.w(0x00, 1 | 16)
    time.sleep(0.05)
    c0 = sample_count(); t0 = time.time()
    time.sleep(seconds)
    c1 = sample_count(); t1 = time.time()
    acq.w(0x00, 1)
    n = (c1 - c0) & 0xFFFFFFFF
    rate = n / (t1 - t0)
    expect = FS_CLK / (1 << dec)
    return rate, expect, rate / expect - 1.0


print('Comparing the two on-board crystals. Neither is a standard.')
print()
print('%10s %16s %16s %14s' % ('interval', 'measured [Hz]', 'expected [Hz]', 'difference'))
runs = []
for secs in (1.0, 10.0, 100.0):
    rate, expect, frac = ratio_over(secs)
    runs.append((secs, frac))
    print('%9.0fs %16.2f %16.2f %13.2f ppm' % (secs, rate, expect, 1e6 * frac))

frac_100 = runs[-1][1]
print()
print('At 100 s the two crystals differ by %.2f ppm.' % (1e6 * frac_100))
print('That is their DISAGREEMENT, not either one\'s error: it could be all of')
print('one, all of the other, or any split between them.')
print()
print('STEP 1: measured (no pass or fail - this is a number, not a verdict)')

---
## Step 2 — Does it move with temperature?

Crystals have a known temperature curve, so if the clock ratio moves as the board warms up, that
is real, correctable, and predicts what a cold morning does to a seven-minute scan.

The Zynq's on-chip temperature sensor is already in the design and Linux exposes it, so this
costs nothing but time.

In [ ]:
import glob

def die_temperature():
    """Zynq on-chip temperature in Celsius, or None if the sensor is not exposed.

    Read from Linux's sysfs rather than from a register, so it needs no bus
    driver. The raw-plus-offset-times-scale form is the standard convention."""
    for base in glob.glob('/sys/bus/iio/devices/iio:device*'):
        try:
            raw = float(open(base + '/in_temp0_raw').read())
            off = float(open(base + '/in_temp0_offset').read())
            sca = float(open(base + '/in_temp0_scale').read())
            return (raw + off) * sca / 1000.0
        except (IOError, OSError, ValueError):
            continue
    return None


t_now = die_temperature()
if t_now is None:
    print('No on-chip temperature sensor exposed by Linux.')
    print('Skipping the temperature correlation; Steps 1 and 3 still stand.')
    TEMP_LOG = []
else:
    print('die temperature now: %.1f C' % t_now)
    print()
    print('Logging the clock ratio against temperature. %d points, ~%.0f minutes.'
          % (12, 12 * 20 / 60.0))
    print('%8s %10s %14s' % ('point', 'temp [C]', 'offset [ppm]'))
    TEMP_LOG = []
    for k in range(12):
        _, _, frac = ratio_over(20.0)
        tc = die_temperature()
        TEMP_LOG.append((tc, 1e6 * frac))
        print('%8d %10.1f %14.2f' % (k, tc, 1e6 * frac))

if len(TEMP_LOG) >= 4:
    tt = np.array([a for a, b in TEMP_LOG])
    pp = np.array([b for a, b in TEMP_LOG])
    if tt.max() - tt.min() > 0.5:
        slope = float(np.polyfit(tt, pp, 1)[0])
        print()
        print('temperature coefficient: %.2f ppm per degree over %.1f C of range'
              % (slope, tt.max() - tt.min()))
        print('over a 7-minute scan a 1 C change would move every frequency by %.2f ppm'
              % abs(slope))
    else:
        print()
        print('the board barely changed temperature (%.1f C) - inconclusive.' % (tt.max() - tt.min()))
        print('Repeat from cold, or after changing the airflow.')

    plt.close('all')
    plt.figure(figsize=(7, 4))
    plt.plot(tt, pp, 'o')
    plt.xlabel('die temperature [C]'); plt.ylabel('clock offset [ppm]')
    plt.title('Does the clock ratio follow temperature?')
    plt.grid(alpha=0.4); plt.tight_layout(); plt.show()
print()
print('STEP 2: logged')

---
## Step 3 — How steady is it, and for how long?

The question every averaging decision depends on: **how long is it worth averaging before drift
overtakes noise?**

Repeat a short measurement many times, then ask how much the answer changes between neighbouring
runs as those runs are grouped into longer and longer blocks. Pure noise falls as the square
root of the averaging time; drift eventually stops it falling and turns it back up. The turning
point is the best averaging time. (This is the Allan deviation; the implementation below is the
standard overlapping form.)

In [ ]:
N_SAMPLES = 120
TAU0      = 5.0        # seconds per point

print('taking %d points of %.0fs each (%.0f minutes)'
      % (N_SAMPLES, TAU0, N_SAMPLES * TAU0 / 60.0))
y = []
for k in range(N_SAMPLES):
    _, _, frac = ratio_over(TAU0)
    y.append(frac)
    if (k + 1) % 20 == 0:
        print('   %d/%d' % (k + 1, N_SAMPLES))
y = np.array(y)


def allan_dev(y, tau0):
    """Overlapping Allan deviation of a series of fractional frequencies.

    For each averaging length m, average the series in blocks of m, then take
    the rms difference between NEIGHBOURING blocks. Differencing neighbours is
    what makes it insensitive to a slow overall trend while still seeing the
    wander that matters."""
    out = []
    for m in range(1, len(y) // 4):
        blocks = np.array([y[i:i + m].mean() for i in range(len(y) - m + 1)])
        d = blocks[m:] - blocks[:-m]
        if len(d) < 2:
            continue
        out.append((m * tau0, float(np.sqrt(0.5 * np.mean(d ** 2)))))
    return np.array(out)


ad = allan_dev(y, TAU0)
print()
print('%12s %16s' % ('averaging [s]', 'stability'))
for tau, s in ad[::max(1, len(ad) // 8)]:
    print('%12.0f %16.2e' % (tau, s))

best = ad[int(np.argmin(ad[:, 1]))]
print()
print('best stability %.2e at an averaging time of %.0f s' % (best[1], best[0]))

plt.close('all')
plt.figure(figsize=(7, 4.4))
plt.loglog(ad[:, 0], ad[:, 1], 'o-', ms=4)
plt.xlabel('averaging time [s]'); plt.ylabel('fractional stability')
plt.title('How long is it worth averaging?')
plt.grid(alpha=0.4, which='both'); plt.tight_layout(); plt.show()
print()
print('STEP 3: measured')

---
## Step 4 — So, is an external reference needed?

Everything above is combined into one answer. The threshold that matters is set by the target:
0.75 Hz at 40 MHz is a fractional stability of 1.9 × 10⁻⁸.

In [ ]:
TARGET_HZ    = 0.75
CARRIER_HZ   = F0
TARGET_FRAC  = TARGET_HZ / CARRIER_HZ

print('=' * 70)
print(' DOES THIS BOARD NEED AN EXTERNAL FREQUENCY REFERENCE?')
print('=' * 70)
print()
print('  target                        : %.2f Hz at %.0f MHz = %.2e fractional'
      % (TARGET_HZ, CARRIER_HZ / 1e6, TARGET_FRAC))
print('  best measured stability       : %.2e at %.0f s' % (best[1], best[0]))
print('  two-crystal disagreement      : %.2f ppm' % (1e6 * frac_100))
print()

stable_enough = best[1] < TARGET_FRAC
print('  stability meets the target    : %s' % ('YES' if stable_enough else 'NO'))
print()
if stable_enough:
    print('  -> For Q, linewidths, shifts and frequency SCATTER, no external')
    print('     reference is needed. Those are ratios, differences and scatters,')
    print('     and the clock error cancels or is negligible in all of them.')
    print()
    print('  -> An external reference is still needed if an ABSOLUTE frequency')
    print('     is ever reported to better than about %.0f ppm.' % (1e6 * frac_100))
else:
    print('  -> The clock wanders more than the target. Either shorten the')
    print('     averaging to %.0f s, control the temperature, or lock to an' % best[0])
    print('     external reference (Stage 14b).')
print()
print('STEP 4: decision made')

---
## Stage 14a summary

| Step | What it delivered |
|---|---|
| 1 | The disagreement between the board's two independent crystals, to better than 0.1 ppm |
| 2 | Whether that disagreement follows temperature, and by how much per degree |
| 3 | How long it is worth averaging before drift overtakes noise |
| 4 | A yes or no on whether Stage 14b is needed |

**What this cannot tell you.** If both crystals are wrong by the same amount, everything here
looks perfect. That is the one question that needs an outside source, and it only matters for
absolute frequencies.

In [ ]:
acq.w(0x00, 0)
print('all quiet')